In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from darts import TimeSeries
from darts.models import TransformerModel
from darts.dataprocessing.transformers import Scaler
from darts.metrics import mae, rmse, r2_score


In [17]:
with open("processed_data_pkl/imputed_training_data.pkl", "rb") as f:
    traffic_dic = pickle.load(f)

with open("processed_data_pkl/weather_global_2014_2025.pkl", "rb") as f:
    weather = pickle.load(f)

air_qual = pd.read_csv("online_data/air_qual/air_qual.csv")

air_qual.drop(columns=["aerosol_optical_depth ()", "dust (μg/m³)"], errors="ignore", inplace=True)
air_qual.dropna(inplace=True)
air_qual.reset_index(inplace=True, drop=True)

air_qual["time"] = pd.to_datetime(air_qual["time"], errors="coerce")
weather["timestamp"] = pd.to_datetime(weather["timestamp"], errors="coerce")

weather = weather[weather["timestamp"].isin(air_qual["time"])]
weather.reset_index(inplace=True, drop=True)

dataset = pd.merge(weather, air_qual, left_on="timestamp", right_on="time", how="inner")


In [18]:
dataset["timestamp"] = pd.to_datetime(dataset["timestamp"])
df = dataset[["timestamp", "pm2_5 (μg/m³)"]].copy()
df.set_index("timestamp", inplace=True)
df = df.sort_index().asfreq("h")

# Convert the pandas dataframe directly into a Darts TimeSeries object
series = TimeSeries.from_dataframe(df, value_cols="pm2_5 (μg/m³)")

In [19]:
train_series, val_series = series.split_before(pd.Timestamp("2025-01-01 00:00:00"))

In [20]:
scaler = Scaler()
train_scaled = scaler.fit_transform(train_series)
val_scaled = scaler.transform(val_series)


INPUT_WINDOW = 24*3
OUTPUT_WINDOW = 24
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

model = TransformerModel(
    input_chunk_length=INPUT_WINDOW,
    output_chunk_length=OUTPUT_WINDOW,
    d_model=64,
    nhead=4,
    num_encoder_layers=3,
    num_decoder_layers=3,
    dim_feedforward=128,
    dropout=0.3,
    activation="relu",
    batch_size=64,
    n_epochs=30,
    optimizer_kwargs={"lr": 1e-3, "weight_decay": 1e-4},
    pl_trainer_kwargs={"accelerator": DEVICE},
    random_state=42
)

/data/ll2531/venv/lib/python3.10/site-packages/torch/random.py:187: UserWarning: CUDA reports that you have 2 available devices, and you have used fork_rng without explicitly specifying which devices are being used. For safety, we initialize *every* CUDA device by default, which can be quite slow if you have a lot of CUDAs. If you know that you are only making use of a few CUDA devices, set the environment variable CUDA_VISIBLE_DEVICES or the 'devices' keyword argument of fork_rng with the set of devices you are actually using. For example, if you are using CPU only, set device.upper()_VISIBLE_DEVICES= or devices=[]; if you are using device 0 only, set CUDA_VISIBLE_DEVICES=0 or devices=[0].  To initialize all devices and suppress this warning, set the 'devices' keyword argument to `range(torch.cuda.device_count())`.
  warnings.warn(message)


In [21]:
print("Training the Darts Transformer Model...")
model.fit(series=train_scaled, val_series=val_scaled)


scaled_forecasts = model.historical_forecasts(
    series=val_scaled,
    start=INPUT_WINDOW,
    forecast_horizon=OUTPUT_WINDOW,
    stride=24,
    retrain=False,
    last_points_only=False
)

true_forecasts = [scaler.inverse_transform(f) for f in scaled_forecasts]
true_targets = [val_series.slice_intersect(f) for f in true_forecasts]

/data/ll2531/venv/lib/python3.10/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(
Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.


hwloc/linux: Ignoring PCI device with non-16bit domain.
Pass --enable-32bits-pci-domain to configure to support such devices
(warning: it would break the library ABI, don't enable unless really needed).
Authorization required, but no authorization protocol specified
Authorization required, but no authorization protocol specified
hwloc/linux: Ignoring PCI device with non-16bit domain.
Pass --enable-32bits-pci-domain to configure to support such devices
(warning: it would break the library ABI, don't enable unless really needed).
Authorization required, but no authorization protocol specified


Training the Darts Transformer Model...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/data/ll2531/venv/lib/python3.10/site-packages/torch/__init__.py:1551: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  return _C._get_float32_matmul_precision()
You are using a CUDA device ('NVIDIA RTX 6000 Ada Generation') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. Fo

Epoch 29: 100%|██████████| 1232/1232 [00:20<00:00, 61.05it/s, train_loss=0.00402, val_loss=0.00351]

`Trainer.fit` stopped: `max_epochs=30` reached.


Epoch 29: 100%|██████████| 1232/1232 [00:20<00:00, 61.05it/s, train_loss=0.00402, val_loss=0.00351]


Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


In [22]:
from darts.metrics import mae, rmse, r2_score

mae_values, rmse_values, r2_values = [], [], []
actual_means = []

for target, forecast in zip(true_targets, true_forecasts):
    mae_values.append(mae(target, forecast))
    rmse_values.append(rmse(target, forecast))
    r2_values.append(r2_score(target, forecast)) 
    
    actual_means.append(target.univariate_values().mean())

global_mae = np.mean(mae_values)
global_rmse = np.mean(rmse_values)
global_r2 = np.mean(r2_values)
global_mean_baseline = np.mean(actual_means)

mae_percentage = (global_mae / global_mean_baseline) * 100
rmse_percentage = (global_rmse / global_mean_baseline) * 100

print(f"Global R² Score   : {global_r2:.4f}")
print(f"MAE% (Normalized) : {mae_percentage:.2f}%")
print(f"RMSE% (Normalized): {rmse_percentage:.2f}%")

Global R² Score   : -10.7176
MAE% (Normalized) : 44.44%
RMSE% (Normalized): 49.37%
